In [399]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_recall_curve, 
                             precision_score, recall_score, f1_score, confusion_matrix, 
                             log_loss, average_precision_score, classification_report)

from preprocessing import import_data
from scoring import score

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings("ignore")

In [400]:
Xt,yt,Xv,yv = import_data()

Importing train/validation data...
Finished the import.


In [401]:
def group_medications(df):
    groups = {
        "biguanide": ["metformin"],
        "meglitinide": ["repaglinide", "nateglinide"],
        "sulfonylurea_gen1": ["chlorpropamide", "acetohexamide", "tolbutamide", "tolazamide"],
        "sulfonylurea_gen2": ["glimepiride", "glipizide", "glyburide"],
        "thiazolidinedione": ["pioglitazone", "rosiglitazone", "troglitazone"],
        "alpha_glucosidase_inhibitor": ["acarbose", "miglitol"],
        "biguanide_sulfonylurea_gen2": ["glyburide-metformin", "glipizide-metformin"],
        "biguanide_thiazolidinedione": ["metformin-rosiglitazone", "metformin-pioglitazone"],
        "sulfonylurea_gen2_thiazolidinedione": ["glimepiride-pioglitazone"],
    }

    result = df.copy()
    priority = ["Up", "Down", "Steady"]

    for group, medications in groups.items():
        columns = df.columns.intersection(medications)
        if len(columns):
            values = df[columns]
            result = result.drop(columns=columns)
            result[group] = np.select([values.eq(status).any(axis=1) for status in priority], priority, default="No")

    return result

Xt=group_medications(Xt)
Xv=group_medications(Xv)

In [402]:
def group_diagnoses(df):
    import pandas as pd

    ranges = {"Circulatory": (390, 459), "Respiratory": (460, 519),
        "Digestive": (520, 579), "Genitourinary": (580, 629),
        "Musculoskeletal": (710, 739), "Neoplasms": (140, 239), "Injury": (800, 999)}
    
    symptoms = {785: "Circulatory", 786: "Respiratory", 787: "Digestive", 788: "Genitourinary"}

    def categorize(value):
        if pd.isna(value):
            return "Missing"

        code = str(value).strip().upper()
        if code in {"", "?", "NAN", "NONE", "MISSING"}:
            return "Missing"

        # Supplementary V/E codes belong to the broad Other category.
        if code.startswith(("V", "E")):
            return "Other"

        try:
            number = float(code)
            if not 1 <= number < 1000:
                return "Other"
            code = int(number)  # 250.13 -> 250, 459.9 -> 459
        except ValueError:
            return "Other"

        if code == 250:
            return "Diabetes"
        if code in symptoms:
            return symptoms[code]
        for group, (lower, upper) in ranges.items():
            if lower <= code <= upper:
                return group
        return "Other"

    result = df.copy()
    for column in ("diag_1", "diag_2", "diag_3", "diaga_3"):
        if column in result.columns:
            result[column] = result[column].map(categorize)

    return result

Xt = group_diagnoses(Xt)
Xv = group_diagnoses(Xv)

In [403]:
num_cols = Xt.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = Xt.select_dtypes(include=['object','category']).columns.tolist()
features = num_cols+cat_cols

In [404]:
Xv

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,payer_code,medical_specialty,diag_1,diag_2,diag_3,max_glu_serum,a1cresult,examide,citoglipton,insulin,change,diabetesmed,biguanide,meglitinide,sulfonylurea_gen1,sulfonylurea_gen2,thiazolidinedione,alpha_glucosidase_inhibitor,biguanide_sulfonylurea_gen2,biguanide_thiazolidinedione,sulfonylurea_gen2_thiazolidinedione
0,2,11,5,13,2,0,1,6,AfricanAmerican,Female,[20-30),NaN,Emergency,Discharged to home,Emergency Room,NaN,NaN,Other,Diabetes,Other,NaN,NaN,No,No,No,No,Yes,No,No,No,Steady,No,No,No,No,No
1,1,51,0,8,0,0,0,5,Caucasian,Male,[40-50),NaN,Emergency,Discharged to home,Emergency Room,NaN,NaN,Neoplasms,Neoplasms,Diabetes,NaN,NaN,No,No,Steady,Ch,Yes,No,No,No,Steady,No,No,No,No,No
2,13,68,2,28,0,0,0,8,Caucasian,Female,[80-90),NaN,Urgent,Discharged to home,Transfer from a hospital,NaN,NaN,Circulatory,Circulatory,Other,NaN,NaN,No,No,Steady,Ch,Yes,No,No,No,Steady,No,No,No,No,No
3,7,62,0,11,0,0,0,7,AfricanAmerican,Male,[60-70),NaN,Urgent,Discharged to home,Transfer from a hospital,NaN,NaN,Neoplasms,Other,Neoplasms,NaN,NaN,No,No,Steady,Ch,Yes,No,No,No,Up,No,No,No,No,No
4,5,35,5,23,0,0,0,9,AfricanAmerican,Male,[70-80),NaN,Emergency,Discharged to home,Emergency Room,NaN,NaN,Circulatory,Circulatory,Circulatory,NaN,NaN,No,No,Steady,No,Yes,No,No,No,No,No,No,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16278,1,1,5,8,0,0,0,4,Caucasian,Male,[40-50),NaN,Elective,Discharged to home,Physician Referral,MD,NaN,Circulatory,Other,Other,NaN,NaN,No,No,Steady,Ch,Yes,No,No,No,No,Steady,No,No,No,No
16279,3,13,1,5,0,0,0,8,Other,Female,[40-50),NaN,Elective,Discharged to home,Physician Referral,HM,NaN,Other,Other,Other,NaN,NaN,No,No,Steady,Ch,Yes,Steady,No,No,Steady,No,No,No,No,No
16280,5,76,1,22,0,1,0,9,Caucasian,Female,[80-90),NaN,Emergency,Discharged to home,Emergency Room,MC,NaN,Other,Other,Other,NaN,NaN,No,No,Up,Ch,Yes,No,No,No,No,No,No,No,No,No
16281,1,1,0,15,3,0,0,7,Caucasian,Male,[80-90),NaN,Emergency,Discharged to home,Emergency Room,MC,NaN,Circulatory,Other,Diabetes,NaN,NaN,No,No,Up,Ch,Yes,No,No,No,No,No,No,No,No,No


In [405]:
def engineer(df):
    df = df.copy()

    visits = [
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
    ]

    # Allow diminishing effects from additional previous visits.
    for column in visits:
        df[f"{column}_log1p"] = np.log1p(df[column])

    df["eng_prior_visits_log1p"] = np.log1p(df[visits].sum(axis=1))
    df["eng_prior_inpatient_any"] = (df["number_inpatient"] > 0).astype(int)
    df["eng_prior_inpatient_2plus"] = (df["number_inpatient"] >= 2).astype(int)

    # Let combined inpatient/emergency utilization have an additional effect.
    df["eng_inpatient_emergency_interaction"] = (
        df["number_inpatient_log1p"]
        * df["number_emergency_log1p"]
    )

    # Distinguish intensity from total volume during the admission.
    days = df["time_in_hospital"].clip(lower=1)
    df["eng_medications_per_day"] = df["num_medications"] / days
    df["eng_labs_per_day"] = df["num_lab_procedures"] / days

    return df

In [406]:
num = Pipeline([
    ("eng", FunctionTransformer(engineer)),
    ("imp", SimpleImputer(strategy="mean")),
    ("scale", StandardScaler()),
])

In [407]:
cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")), 
                ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=50))])

In [408]:
prep = ColumnTransformer([("num", num, num_cols),
                          ("cat", cat, cat_cols)])

In [409]:
clf = Pipeline([("prep", prep), 
                ("model", LogisticRegression(class_weight={0: 1, 1: 14}, 
                                             max_iter=1250,
                                             C=0.50, 
                                             random_state=42)
                                             )])

In [410]:
# Fit the model to the training data
clf.fit(Xt,yt)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [411]:
probs = clf.predict_proba(Xv)[:,1]
score(yv, y_pred=None, proba=probs, threshold=0.55631)

Accuracy: 0.481
Confusion matrix:
 [[6428 8038]
 [ 414 1403]]
              precision    recall  f1-score   support

           0       0.94      0.44      0.60     14466
           1       0.15      0.77      0.25      1817

    accuracy                           0.48     16283
   macro avg       0.54      0.61      0.43     16283
weighted avg       0.85      0.48      0.56     16283

F1 @0.55631: 0.249
F2 @0.55631: 0.42
ROC-AUC: 0.6736
PR-AUC:  0.2143

Best-threshold F1: 0.28547 at threshold 0.68876.

Best-threshold F2: 0.42353 at threshold 0.51924.


In [412]:
# The following code reports the weights and the feature names.

prep, model = clf.named_steps["prep"], clf.named_steps["model"]

def names_from(t, c):
    c = list(Xt.columns[c]) if isinstance(c, slice) else [c] if isinstance(c, str) else list(c)
    for s in reversed([s for _, s in t.steps] if hasattr(t, "steps") else [t]):
        for m in ("get_feature_names_out", "get_feature_names"):
            for a in ((c,), ()):
                try: return list(getattr(s, m)(*a))
                except Exception: pass
    return c  # passthrough / 1:1 transformer

names = [n for k, t, c in prep.transformers_ if not (isinstance(t, str) and t == "drop")
         for n in names_from(t, [Xt.columns[i] for i in c] if k == "remainder" and isinstance(t, str) else c)]

print(f"Model Intercept (Bias): {model.intercept_[0]:.4f}\n")

weights_df = pd.DataFrame({"Feature Name": names, "Weight (Coefficient)": model.coef_[0]}).sort_values("Weight (Coefficient)", ascending=False, ignore_index=True)
display(weights_df)

Model Intercept (Bias): -0.0643



,Feature Name,Weight (Coefficient)
0,discharge_disposition_id_Discharged/transferre...,1.828043
1,medical_specialty_Hematology,1.517098
2,discharge_disposition_id_Discharged/transferre...,1.377471
3,discharge_disposition_id_Discharged/transferre...,1.081470
4,medical_specialty_Hematology/Oncology,0.807139
...,...,...
198,discharge_disposition_id_Hospice / home,-0.862673
199,medical_specialty_Otolaryngology,-0.915788
200,medical_specialty_Pediatrics-Endocrinology,-1.021335
201,age_[0-10),-1.041163


In [413]:
weights_df.sort_values("Weight (Coefficient)", key=abs, ascending=False, ignore_index=True).head(20)

,Feature Name,Weight (Coefficient)
0,discharge_disposition_id_Expired,-5.293272
1,discharge_disposition_id_Discharged/transferre...,1.828043
2,medical_specialty_Hematology,1.517098
3,discharge_disposition_id_Discharged/transferre...,1.377471
4,discharge_disposition_id_Discharged/transferre...,1.081470
5,age_[0-10),-1.041163
6,medical_specialty_Pediatrics-Endocrinology,-1.021335
7,medical_specialty_Otolaryngology,-0.915788
8,discharge_disposition_id_Hospice / home,-0.862673
9,medical_specialty_Hematology/Oncology,0.807139


In [414]:
weights_df.sort_values("Weight (Coefficient)", key=abs, ascending=True, ignore_index=True).head(20)

,Feature Name,Weight (Coefficient)
0,admission_type_id_Elective,0.000734
1,x1,-0.001799
2,admission_type_id_Not Mapped,0.004626
3,medical_specialty_Surgery-General,0.006217
4,biguanide_Steady,-0.006257
5,x10,0.007429
6,change_No,-0.008170
7,thiazolidinedione_No,0.008992
8,biguanide_sulfonylurea_gen2_No,-0.010390
9,x16,-0.010414
